# Step 1: Load and Prepare the Red Wine Quality Dataset

Load the UCI Red Wine Quality data, engineer three domain features, create a binary quality target, and save the resulting 14-feature dataset. This step verifies dataset dimensions, missing values, feature names, and target class counts.

In [1]:
import pandas as pd

DATASET_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "wine-quality/winequality-red.csv"
)
OUTPUT_PATH = "wine_dataset_cleaned.csv"

df = pd.read_csv(DATASET_URL, sep=";")
df.columns = df.columns.str.replace(" ", "_", regex=False)

df["total_acidity"] = df["fixed_acidity"] + df["volatile_acidity"]
df["bound_sulfur_dioxide"] = (
    df["total_sulfur_dioxide"] - df["free_sulfur_dioxide"]
)
df["sugar_to_acidity_ratio"] = df["residual_sugar"] / (
    df["total_acidity"] + 1e-5
)
df["quality_label"] = (df["quality"] >= 6).astype(int)
df = df.drop(columns=["quality"])

feature_columns = [column for column in df.columns if column != "quality_label"]
df.to_csv(OUTPUT_PATH, index=False)

print(f"Total rows: {len(df)}")
print(f"Total input features: {len(feature_columns)}")
print(f"Missing value count: {int(df.isna().sum().sum())}")
print("Feature names:")
for feature in feature_columns:
    print(f"- {feature}")
print("Target class counts:")
print(df["quality_label"].value_counts().sort_index().to_string())
print(f"Saved cleaned dataset to: {OUTPUT_PATH}")

assert len(df) == 1599, f"Expected 1599 rows, got {len(df)}"
assert len(feature_columns) == 14, (
    f"Expected 14 input features, got {len(feature_columns)}"
)

Total rows: 1599
Total input features: 14
Missing value count: 0
Feature names:
- fixed_acidity
- volatile_acidity
- citric_acid
- residual_sugar
- chlorides
- free_sulfur_dioxide
- total_sulfur_dioxide
- density
- pH
- sulphates
- alcohol
- total_acidity
- bound_sulfur_dioxide
- sugar_to_acidity_ratio
Target class counts:
quality_label
0    744
1    855
Saved cleaned dataset to: wine_dataset_cleaned.csv


## Step 2: Stratified Train/Test Split and Feature Scaling

Separate the 14 input features from `quality_label`, then create a reproducible 80/20 split using stratification so both partitions retain similar class proportions. Export the held-out test features in their original, unscaled form with the target column. Fit `StandardScaler` using only the training features to avoid test-data leakage, and persist the fitted scaler in `model/scaler.pkl` for later application use. No classifier is trained in this step.

In [4]:
from pathlib import Path

import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "quality_label"

feature_columns = [column for column in df.columns if column != TARGET_COLUMN]
X = df[feature_columns]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Keep the exported test set unscaled for consistent preprocessing in the app.
test_data = X_test.copy()
test_data[TARGET_COLUMN] = y_test
test_data.to_csv("test_data.csv", index=False)

# Fit preprocessing only on training data to prevent test-set leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_dir = Path("model")
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, model_dir / "scaler.pkl")

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Input features: {len(feature_columns)}")
print("Training target proportions:")
print(y_train.value_counts(normalize=True).sort_index().to_string())
print("Test target proportions:")
print(y_test.value_counts(normalize=True).sort_index().to_string())
print(f"Exported unscaled test data: {test_data.shape}")
print(f"Scaler fitted on training rows only: {scaler.n_samples_seen_ == len(X_train)}")
print(f"Saved scaler: {model_dir / 'scaler.pkl'}")

assert len(X_train) == 1279
assert len(X_test) == 320
assert test_data.columns.tolist() == feature_columns + [TARGET_COLUMN]
assert scaler.n_samples_seen_ == len(X_train)
assert test_data.isna().sum().sum() == 0

Training rows: 1279
Test rows: 320
Input features: 14
Training target proportions:
quality_label
0    0.465207
1    0.534793
Test target proportions:
quality_label
0    0.465625
1    0.534375
Exported unscaled test data: (320, 15)
Scaler fitted on training rows only: True
Saved scaler: model/scaler.pkl
